# Arrays, Strings and Hashing

Notebook for **Chapter 2** of the *dsa* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/dsa/ch02.html

Tip: change a number, predict the output, then run the cell again.

## 2.1 Arrays and In-Place Tricks

In [ ]:
def keep_in_place(a, keep):
    """Keep only items where keep(item) is True, in the same order.
    Returns how many were kept. The list a is changed in place."""
    w = 0                      # writer: next free slot for a kept item
    for r in range(len(a)):    # reader: looks at every item once
        if keep(a[r]):         # decide: is this item kept?
            a[w] = a[r]        # copy it to the writer slot
            w += 1             # advance the writer only when we keep
    return w                   # a[:w] is the answer; a[w:] is leftover

nums = [5, 2, 9, 2, 7]
k = keep_in_place(nums, lambda x: x != 2)
print(k, nums[:k])

3 [5, 9, 7]


In [ ]:
def move_zeros(a):
    w = 0
    for r in range(len(a)):
        if a[r] != 0:
            a[w] = a[r]
            w += 1
    for i in range(w, len(a)):   # fill the tail with zeros
        a[i] = 0

for case in ([0, 1, 0, 3, 12], [], [0, 0], [4, 5]):
    move_zeros(case)
    print(case)
a = [0, 1, 0, 3, 12]
move_zeros(a)
assert a == [1, 3, 12, 0, 0]

[1, 3, 12, 0, 0]
[]
[0, 0]
[4, 5]


In [ ]:
def first_missing_positive(a):
    n = len(a)
    for i in range(n):
        # keep swapping a[i] to its home until it is out of range or already home
        while 1 <= a[i] <= n and a[a[i] - 1] != a[i]:
            home = a[i] - 1
            a[i], a[home] = a[home], a[i]
    for i in range(n):
        if a[i] != i + 1:
            return i + 1
    return n + 1

print(first_missing_positive([3, 4, -1, 1]))
print(first_missing_positive([1, 2, 0]))
print(first_missing_positive([7, 8, 9]))
print(first_missing_positive([1, 1, 1]))
assert first_missing_positive([1, 2, 3]) == 4

2
3
1
2


## 2.2 Strings

In [ ]:
s = "python"
try:
    s[0] = "P"                 # strings cannot be changed
except TypeError as e:
    print("error:", e)
print(s[1:4], s[-1], s[::-1], s.upper(), s)   # upper() returns a NEW string
print(ord("a"), ord("c") - ord("a"), chr(ord("a") + 2))
words = ["fast", "dsa", "wins"]
print(" ".join(words))         # build big strings with join, not += in a loop

error: 'str' object does not support item assignment
yth n nohtyp PYTHON python
97 2 c
fast dsa wins


In [ ]:
def is_palindrome(text):
    # step 1: keep only letters and digits, lower-case them
    clean = [c.lower() for c in text if c.isalnum()]
    l, r = 0, len(clean) - 1          # two pointers at the ends
    while l < r:                      # stop when they meet or cross
        if clean[l] != clean[r]:      # first mismatch decides
            return False
        l += 1                        # move both inwards
        r -= 1
    return True

print(is_palindrome("Madam, I'm Adam"), is_palindrome("Race a car"))

True False


In [ ]:
def longest_pal(s):
    best_l, best_r = 0, 0
    def expand(l, r):
        while l >= 0 and r < len(s) and s[l] == s[r]:
            l -= 1
            r += 1
        return l + 1, r - 1               # the last valid window
    for c in range(len(s)):
        for l, r in (expand(c, c), expand(c, c + 1)):   # odd, even centre
            if r - l > best_r - best_l:
                best_l, best_r = l, r
    return s[best_l:best_r + 1]

print(longest_pal("babad"), longest_pal("cbbd"), longest_pal("abc"))
assert longest_pal("aaaa") == "aaaa"

bab bb a


In [ ]:
a = "é"          # e with acute accent as ONE code point
b = "é"         # plain e followed by a combining accent
print(len(a), len(b), a == b)
import unicodedata
print(unicodedata.normalize("NFC", b) == a)

1 2 False
True


## 2.3 Hash Maps and Sets: How They Work Inside

In [ ]:
class ToyMap:
    def __init__(self, size=8):
        self.buckets = [[] for _ in range(size)]
    def _index(self, key):                       # the hash function
        return sum(ord(c) for c in key) % len(self.buckets)
    def put(self, key, value):
        bucket = self.buckets[self._index(key)]
        for pair in bucket:                      # key already there? update it
            if pair[0] == key:
                pair[1] = value
                return
        bucket.append([key, value])              # else add to the chain
    def get(self, key, default=None):
        for k, v in self.buckets[self._index(key)]:
            if k == key:
                return v
        return default

m = ToyMap()
for i, k in enumerate(["cat", "dog", "sun", "fig", "ant"]):
    m.put(k, i)
print(m.buckets)
assert m.get("fig") == 3 and m.get("zzz") is None
m.put("fig", 99)
assert m.get("fig") == 99

class BadMap(ToyMap):                            # every key goes to bucket 0
    def _index(self, key):
        return 0

keys = ["k" + str(i) for i in range(10)]
good, bad = ToyMap(), BadMap()
for k in keys:
    good.put(k, 1)
    bad.put(k, 1)
print("longest chain:", max(len(b) for b in good.buckets), max(len(b) for b in bad.buckets))

[[['cat', 0]], [], [['dog', 1]], [['ant', 4]], [], [], [['sun', 2], ['fig', 3]], []]
longest chain: 2 10


In [ ]:
d = {}
try:
    d[[1, 2]] = "list key"            # lists can change, so they are unhashable
except TypeError as e:
    print("error:", e)
d[(1, 2)] = "tuple key"               # tuples of numbers/text are fine
print(d[(1, 2)])
print(0.1 + 0.2 == 0.3, (0.1 + 0.2) in {0.3})   # floats make poor keys

class Pt:                              # a key object that can be mutated
    def __init__(self, x):
        self.x = x
    def __hash__(self):
        return hash(self.x)
    def __eq__(self, other):
        return self.x == other.x

p = Pt(1)
table = {p: "found"}
print(p in table)
p.x = 2                                # change the key after inserting it
print(p in table, Pt(1) in table, len(table))

error: cannot use 'list' as a dict key (unhashable type: 'list')
tuple key
False False
True
False False 1


## 2.4 Frequency Counting

In [ ]:
from collections import Counter

def freq(items):
    counts = {}                                 # manual version
    for x in items:
        counts[x] = counts.get(x, 0) + 1        # missing key counts as 0
    return counts

text = "mississippi"
print(freq(text))
print(Counter(text))                            # same idea, built in
print(Counter(text).most_common(2))
assert freq(text) == dict(Counter(text))

{'m': 1, 'i': 4, 's': 4, 'p': 2}
Counter({'i': 4, 's': 4, 'p': 2, 'm': 1})
[('i', 4), ('s', 4)]


In [ ]:
from collections import Counter

def first_unique(s):
    counts = Counter(s)
    for i, c in enumerate(s):
        if counts[c] == 1:
            return i
    return -1

print(first_unique("leetcode"), first_unique("loveleetcode"), first_unique("aabb"), first_unique(""))

0 2 -1 -1


In [ ]:
from collections import Counter

def find_anagrams(s, p):
    need, window, res, m = Counter(p), Counter(), [], len(p)
    for i, c in enumerate(s):
        window[c] += 1                         # letter enters on the right
        if i >= m:
            left = s[i - m]                    # letter leaves on the left
            window[left] -= 1
            if window[left] == 0:
                del window[left]               # keep dictionaries comparable
        if window == need:
            res.append(i - m + 1)
    return res

print(find_anagrams("cbaebabacd", "abc"), find_anagrams("abab", "ab"), find_anagrams("a", "ab"))

[0, 6] [0, 1, 2] []


## 2.5 Duplicates and Set Membership

In [ ]:
def first_repeat(items):
    seen = set()                    # guest list
    for x in items:
        if x in seen:               # already on the list: a repeat
            return x
        seen.add(x)                 # first time: write the name down
    return None

print(first_repeat([4, 1, 2, 4, 5]), first_repeat([1, 2, 3]))

4 None


In [ ]:
def near_duplicate(a, k):
    last = {}
    for i, x in enumerate(a):
        if x in last and i - last[x] <= k:
            return True
        last[x] = i                      # always remember the latest position
    return False

print(near_duplicate([4, 1, 2, 4, 5], 3), near_duplicate([1, 2, 3, 1, 2, 3], 2), near_duplicate([1, 0, 1, 1], 1))

True False True


In [ ]:
def longest_unique(s):
    last, left, best = {}, 0, 0
    for i, c in enumerate(s):
        if c in last and last[c] >= left:     # repeat INSIDE the window
            left = last[c] + 1
        last[c] = i
        best = max(best, i - left + 1)
    return best

print(longest_unique("abcabcbb"), longest_unique("bbbbb"), longest_unique("abba"), longest_unique(""))

3 1 2 0


## 2.6 Grouping with a Canonical Key

In [ ]:
from collections import defaultdict

def group_by(items, key):
    groups = defaultdict(list)          # missing key starts as an empty list
    for x in items:
        groups[key(x)].append(x)        # same key, same bucket
    return groups

g = group_by(["eat", "tea", "tan", "ate", "nat", "bat"], lambda w: "".join(sorted(w)))
print(dict(g))
assert sorted(sorted(v) for v in g.values()) == [["ate", "eat", "tea"], ["bat"], ["nat", "tan"]]

{'aet': ['eat', 'tea', 'ate'], 'ant': ['tan', 'nat'], 'abt': ['bat']}


In [ ]:
from collections import defaultdict

def valid_board(board):
    rows, cols, boxes = defaultdict(set), defaultdict(set), defaultdict(set)
    for r in range(9):
        for c in range(9):
            v = board[r][c]
            if v == 0:
                continue                                  # empty cell
            b = (r // 3, c // 3)                          # which box
            if v in rows[r] or v in cols[c] or v in boxes[b]:
                return False
            rows[r].add(v); cols[c].add(v); boxes[b].add(v)
    return True

board = [[(r * 3 + r // 3 + c) % 9 + 1 for c in range(9)] for r in range(9)]
print(valid_board(board))
board[0][1] = board[0][0]                                 # repeat a digit in row 0
print(valid_board(board))
print(valid_board([[0] * 9 for _ in range(9)]))

True
False
True


## 2.7 The Two-Sum Family

In [ ]:
def two_sum(nums, target):
    seen = {}                          # value -> index where we saw it
    for i, x in enumerate(nums):
        need = target - x              # the partner we are looking for
        if need in seen:               # partner already arrived?
            return [seen[need], i]
        seen[x] = i                    # check FIRST, then store
    return []

print(two_sum([3, 8, 5, 2, 11], 16), two_sum([3, 3], 6), two_sum([3, 4], 6))
assert two_sum([2, 7, 11, 15], 9) == [0, 1]

[2, 4] [0, 1] []


In [ ]:
def three_sum(nums):
    nums = sorted(nums)
    res = []
    for i in range(len(nums) - 2):
        if i > 0 and nums[i] == nums[i - 1]:
            continue                               # skip repeated first number
        l, r = i + 1, len(nums) - 1
        while l < r:
            s = nums[i] + nums[l] + nums[r]
            if s < 0:
                l += 1                             # need a bigger sum
            elif s > 0:
                r -= 1                             # need a smaller sum
            else:
                res.append([nums[i], nums[l], nums[r]])
                l += 1
                r -= 1
                while l < r and nums[l] == nums[l - 1]:
                    l += 1                         # skip repeated second number
    return res

print(three_sum([-1, 0, 1, 2, -1, -4]), three_sum([0, 0, 0, 0]), three_sum([1, 2]))

[[-1, -1, 2], [-1, 0, 1]] [[0, 0, 0]] []


## 2.8 Prefix Sums with a Dictionary

In [ ]:
def count_subarrays(nums, k):
    counts = {0: 1}               # the empty prefix: a total of 0 seen once
    running, answer = 0, 0
    for x in nums:
        running += x              # prefix sum so far
        answer += counts.get(running - k, 0)   # earlier prefixes that are k lower
        counts[running] = counts.get(running, 0) + 1   # record this prefix
    return answer

print(count_subarrays([1, 2, 1, 2, 1], 3))

def brute(nums, k):                  # slow but obviously correct
    return sum(1 for i in range(len(nums)) for j in range(i, len(nums)) if sum(nums[i:j + 1]) == k)

import random
random.seed(7)
for _ in range(200):
    a = [random.randint(-3, 3) for _ in range(random.randint(0, 9))]
    k = random.randint(-3, 3)
    assert count_subarrays(a, k) == brute(a, k)
print(count_subarrays([1, 1, 1], 2), count_subarrays([3, 4, -7, 1, 3, 3, 1, -4], 7), count_subarrays([0, 0, 0], 0))

4
2 4 6


In [ ]:
def count_subarrays(nums, k):       # same as the template
    counts, running, answer = {0: 1}, 0, 0
    for x in nums:
        running += x
        answer += counts.get(running - k, 0)
        counts[running] = counts.get(running, 0) + 1
    return answer

def count_submatrices(grid, target):
    R, C, total = len(grid), len(grid[0]), 0
    for top in range(R):
        col = [0] * C                      # column sums of rows top..bottom
        for bottom in range(top, R):
            for c in range(C):
                col[c] += grid[bottom][c]
            total += count_subarrays(col, target)
    return total

g = [[0, 1, 0], [1, 1, 1], [0, 1, 0]]
assert count_submatrices(g, 3) == 6        # middle row, middle column, four 2x2 blocks
assert count_submatrices([[1, -1], [-1, 1]], 0) == 5
print(count_submatrices([[0, 1, 0], [1, 1, 1], [0, 1, 0]], 0))

4


## 2.9 Longest Consecutive Sequence

In [ ]:
def longest_consecutive(nums):
    pool = set(nums)                 # O(1) membership, also removes repeats
    best = 0
    for n in pool:
        if n - 1 not in pool:        # n begins a run
            length = 1
            while n + length in pool:    # walk forward
                length += 1
            best = max(best, length)
    return best

print(longest_consecutive([100, 4, 200, 1, 3, 2]))
print(longest_consecutive([1, 2, 2, 3]), longest_consecutive([]), longest_consecutive([-2, -1, 0, 5]), longest_consecutive([0, 3, 7, 2, 5, 8, 4, 6, 0, 1]))

4
3 0 3 9


## Chapter recap

## Practice set

Try each problem before opening the answer. They climb from easy to hard. Problems 5, 6, 15 and 20 are predict-the-output or interview style. Problems 13 and 14 are find-the-bug.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/dsa/ch02.html